# Ray Cluster Connection Template

This notebook connects to your cloud Ray cluster for distributed computing.

In [1]:
import ray
import pyarrow.fs
from ray import tune
from ray.train import RunConfig

## 1. Port Forward Ray Cluster

In a terminal, run:
```bash
kubectl port-forward -n ray-system svc/raycluster-sample-head-svc 10001:10001 8265:8265
```

In [2]:
# Connect to Ray cluster
ray.init("ray://localhost:10001")

# Verify connection
print(f"Connected to Ray cluster!")
print(f"\nAvailable resources: {ray.available_resources()}")

# Check cluster nodes
print(f"\nCluster nodes:")
for node in ray.nodes():
    print(f"  Node: {node['NodeName']}")
    print(f"    Alive: {node['Alive']}")
    print(f"    Resources: {node['Resources']}")
    print()

2025-10-26 14:33:52,296	INFO client_builder.py:241 -- Passing the following kwargs to ray.init() on the server: log_to_driver
SIGTERM handler is not set because current thread is not the main thread.


Connected to Ray cluster!

Available resources: {'node:10.244.122.7': 1.0, 'object_store_memory': 6256174693.0, 'memory': 22548578304.0, 'node:10.244.4.204': 1.0, 'node:10.244.19.135': 1.0, 'node:10.244.211.7': 1.0, 'node:10.244.117.199': 1.0, 'CPU': 10.0, 'node:__internal_head__': 1.0}

Cluster nodes:
  Node: 10.244.211.7
    Alive: True
    Resources: {'node:10.244.211.7': 1.0, 'CPU': 2.0, 'memory': 4294967296.0, 'object_store_memory': 1236865843.0}

  Node: 10.244.117.199
    Alive: True
    Resources: {'CPU': 2.0, 'node:10.244.117.199': 1.0, 'memory': 4294967296.0, 'object_store_memory': 1232240640.0}

  Node: 10.244.122.7
    Alive: True
    Resources: {'CPU': 2.0, 'node:10.244.122.7': 1.0, 'memory': 4294967296.0, 'object_store_memory': 1236858470.0}

  Node: 10.244.19.135
    Alive: True
    Resources: {'node:__internal_head__': 1.0, 'node:10.244.19.135': 1.0, 'CPU': 2.0, 'memory': 5368709120.0, 'object_store_memory': 1318746931.0}

  Node: 10.244.4.204
    Alive: True
    Resour

## 2. Setup MinIO Storage (for results)

In [3]:
# Setup MinIO S3 filesystem
s3_fs = pyarrow.fs.S3FileSystem(
    endpoint_override="localhost:9000",  # Port forward: kubectl port-forward -n minio svc/minio 9000:9000
    scheme="http",
    access_key="minioadmin",
    secret_key="minioadmin123",
    allow_bucket_creation=True
)

## 3. Test Remote Function

In [4]:
@ray.remote
def hello_ray():
    import socket
    return f"Hello from {socket.gethostname()}!"

# Run on cluster
result = ray.get(hello_ray.remote())
print(result)

Hello from raycluster-sample-head-vctz7!


## 4. Your Training Code Here

In [5]:
# Optional: Continuous Workload Generator
# Run this to keep the cluster busy for extended monitoring

import time

@ray.remote(num_cpus=1)  # Request 1 CPU per task
def heavy_computation(duration_seconds):
    """
    Keep CPU busy for specified duration (no external deps needed)
    """
    import time
    import random
    import math
    import socket
    
    hostname = socket.gethostname()
    start = time.time()
    iterations = 0
    
    while time.time() - start < duration_seconds:
        # CPU intensive work with standard library only
        # Compute lots of random calculations
        values = [random.random() * 1000 for _ in range(10000)]
        
        # Do expensive math operations
        result = sum([math.sqrt(abs(v)) * math.sin(v) * math.cos(v) for v in values])
        
        # More CPU work
        matrix = [[random.random() for _ in range(100)] for _ in range(100)]
        computed = [[sum(row) for row in matrix] for _ in range(10)]
        
        iterations += 1
    
    return f"{hostname}: Completed {iterations} iterations in {duration_seconds}s"

# Uncomment to run continuous workload
# print("Submitting continuous workload...")
# print("This will keep 20 tasks running for 60 seconds each")
# print("Check Grafana to see CPU/memory metrics!")
# 
# futures = [heavy_computation.remote(60) for _ in range(20)]
# results = ray.get(futures)
# print("\n✓ Workload completed!")
# for r in results:
#     print(f"  - {r}")

In [6]:
# Optional: Continuous Workload Generator
# Run this to keep the cluster busy for extended monitoring

import time

@ray.remote
def heavy_computation(duration_seconds):
    """
    Keep CPU busy for specified duration
    """
    import numpy as np
    import time
    
    start = time.time()
    while time.time() - start < duration_seconds:
        # Matrix operations - CPU intensive
        a = np.random.rand(500, 500)
        b = np.random.rand(500, 500)
        c = np.dot(a, b)
        # More CPU work
        result = np.linalg.svd(c)
    
    return f"Completed {duration_seconds}s of work"

# Uncomment to run continuous workload
# print("Submitting continuous workload...")
# print("This will keep 20 tasks running for 60 seconds each")
# print("Check Grafana to see CPU/memory metrics!")
# 
# futures = [heavy_computation.remote(60) for _ in range(20)]
# results = ray.get(futures)
# print("✓ Workload completed!")

## 5. CPU-Intensive Workload Test

Run this to stress test the cluster and see metrics in Grafana!

In [7]:
# Don't forget to shutdown when done
ray.shutdown()